In [7]:
import requests
import json

# 1) 본인 TMDB API 키 입력
API_KEY = "567860aad80693f35f9ae5193ec2e291"   # <- 여기만 바꿔줘!
BASE_URL = "https://api.themoviedb.org/3"

# 2) 네가 직접 골라둔 작품 "이름" 리스트 (영화/TV 섞여 있어도 됨)
#    TMDB에서 보이는 한국어 제목 위주로 적는 걸 추천
SELECTED_TITLES = [
"천문: 하늘에 묻는다",
"왕의 남자",
"해어화",
"명량",
"관상",
"남한산성",
"이강에는 달이 흐른다",
"폭군의 셰프",
"원경",
"원경: 단오의 인연",
"춘화연애담",
"옥씨부인전",
"정년이",
"우씨왕후",
"세작, 매혹된 자들",
"백일의 낭군님",
"성균관 스캔들"
]

# ---------------- 공통 유틸 함수 ---------------- #

def search_best_match(title, kind):
    """
    title로 TMDB에서 검색해서, TV/영화 중 가장 적절한 1개를 골라 반환.
    kind: "tv" 또는 "movie"
    """
    if kind == "tv":
        url = f"{BASE_URL}/search/tv"
    else:
        url = f"{BASE_URL}/search/movie"

    resp = requests.get(
        url,
        params={
            "api_key": API_KEY,
            "query": title,
            "language": "ko-KR",
            "region": "KR",
            "include_adult": "false",
        },
    )
    resp.raise_for_status()
    results = resp.json().get("results", [])

    if not results:
        return None

    # 1차: 한국어/한국 작품 우선
    ko_like = []
    others = []
    for r in results:
        lang = r.get("original_language")
        origin_country = r.get("origin_country") or []
        if lang == "ko" or "KR" in origin_country:
            ko_like.append(r)
        else:
            others.append(r)

    candidates = ko_like if ko_like else results

    # 2차: popularity가 가장 높은 것 선택
    best = max(candidates, key=lambda x: x.get("popularity", 0) or 0)
    return best


def fetch_watch_providers(kind, tmdb_id):
    """
    해당 작품의 watch providers 정보에서 TVING 여부 확인
    kind: "tv" 또는 "movie"
    """
    url = f"{BASE_URL}/{kind}/{tmdb_id}/watch/providers"
    resp = requests.get(url, params={"api_key": API_KEY})
    resp.raise_for_status()
    data = resp.json()

    providers_kr = data.get("results", {}).get("KR", {})
    flatrate = providers_kr.get("flatrate", []) or []

    on_tving = False
    tving_provider_id = None

    for p in flatrate:
        if "TVING" in (p.get("provider_name") or "").upper():
            on_tving = True
            tving_provider_id = p.get("provider_id")
            break

    return on_tving, tving_provider_id


def fetch_detail(kind, tmdb_id):
    """
    TV/영화 상세 정보 가져오기
    kind: "tv" 또는 "movie"
    """
    if kind == "tv":
        url = f"{BASE_URL}/tv/{tmdb_id}"
    else:
        url = f"{BASE_URL}/movie/{tmdb_id}"

    resp = requests.get(
        url,
        params={
            "api_key": API_KEY,
            "language": "ko-KR",
        },
    )
    resp.raise_for_status()
    detail = resp.json()

    on_tving, tving_provider_id = fetch_watch_providers(kind, tmdb_id)

    if kind == "tv":
        return {
            "kind": "tv",
            "id": detail.get("id"),
            "name": detail.get("name"),
            "original_name": detail.get("original_name"),
            "overview": detail.get("overview"),
            "first_air_date": detail.get("first_air_date"),
            "poster_path": detail.get("poster_path"),
            "backdrop_path": detail.get("backdrop_path"),
            "vote_average": detail.get("vote_average"),
            "vote_count": detail.get("vote_count"),
            "genres": detail.get("genres"),  # [{id, name}, ...]
            "origin_country": detail.get("origin_country"),
            "on_tving_kr": on_tving,
            "tving_provider_id": tving_provider_id,
        }
    else:
        return {
            "kind": "movie",
            "id": detail.get("id"),
            "title": detail.get("title"),
            "original_title": detail.get("original_title"),
            "overview": detail.get("overview"),
            "release_date": detail.get("release_date"),
            "poster_path": detail.get("poster_path"),
            "backdrop_path": detail.get("backdrop_path"),
            "vote_average": detail.get("vote_average"),
            "vote_count": detail.get("vote_count"),
            "genres": detail.get("genres"),
            "original_language": detail.get("original_language"),
            "on_tving_kr": on_tving,
            "tving_provider_id": tving_provider_id,
        }

# ---------------- 메인 로직 ---------------- #

results = []

for title in SELECTED_TITLES:
    print(f"\n=== '{title}' 검색 중 ===")

    best_tv = search_best_match(title, "tv")
    best_movie = search_best_match(title, "movie")

    chosen = None
    chosen_kind = None

    # 우선순위 로직:
    # 1) 둘 다 있으면 popularity 더 높은 쪽 선택
    # 2) 하나만 있으면 그쪽 선택
    if best_tv and best_movie:
        pop_tv = best_tv.get("popularity", 0) or 0
        pop_movie = best_movie.get("popularity", 0) or 0
        if pop_tv >= pop_movie:
            chosen = best_tv
            chosen_kind = "tv"
        else:
            chosen = best_movie
            chosen_kind = "movie"
    elif best_tv:
        chosen = best_tv
        chosen_kind = "tv"
    elif best_movie:
        chosen = best_movie
        chosen_kind = "movie"

    if not chosen:
        print(f"  -> '{title}' 에 해당하는 결과를 찾지 못했습니다.")
        continue

    tmdb_id = chosen["id"]
    print(f"  -> 선택된 {chosen_kind.upper()} id={tmdb_id}, name={chosen.get('name') or chosen.get('title')}")

    try:
        detail = fetch_detail(chosen_kind, tmdb_id)
        results.append(detail)
        print(f"  -> 상세 정보 수집 완료 (on_tving_kr={detail['on_tving_kr']})")
    except Exception as e:
        print(f"  -> 상세 정보 수집 실패:", e)

print("\n총 수집된 작품 수:", len(results))

# 20개만 필요하면 앞에서부터 잘라 쓰기 (필요 없으면 이 줄 주석 처리)
results = results[:20]

OUTPUT_FILE = "musetiv_tving_historical_20_by_title.json"
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("저장 완료:", OUTPUT_FILE)



=== '천문: 하늘에 묻는다' 검색 중 ===
  -> 선택된 MOVIE id=569267, name=천문: 하늘에 묻는다
  -> 상세 정보 수집 완료 (on_tving_kr=True)

=== '왕의 남자' 검색 중 ===
  -> 선택된 MOVIE id=45035, name=왕의 남자
  -> 상세 정보 수집 완료 (on_tving_kr=True)

=== '해어화' 검색 중 ===
  -> 선택된 MOVIE id=385261, name=해어화
  -> 상세 정보 수집 완료 (on_tving_kr=True)

=== '명량' 검색 중 ===
  -> 선택된 MOVIE id=282631, name=명량
  -> 상세 정보 수집 완료 (on_tving_kr=True)

=== '관상' 검색 중 ===
  -> 선택된 MOVIE id=220176, name=관상
  -> 상세 정보 수집 완료 (on_tving_kr=False)

=== '남한산성' 검색 중 ===
  -> 선택된 MOVIE id=437081, name=남한산성
  -> 상세 정보 수집 완료 (on_tving_kr=True)

=== '이강에는 달이 흐른다' 검색 중 ===
  -> 선택된 TV id=274570, name=이강에는 달이 흐른다
  -> 상세 정보 수집 완료 (on_tving_kr=True)

=== '폭군의 셰프' 검색 중 ===
  -> 선택된 TV id=280945, name=폭군의 셰프
  -> 상세 정보 수집 완료 (on_tving_kr=True)

=== '원경' 검색 중 ===
  -> 선택된 TV id=226103, name=원경
  -> 상세 정보 수집 완료 (on_tving_kr=False)

=== '원경: 단오의 인연' 검색 중 ===
  -> 선택된 TV id=282101, name=원경: 단오의 인연
  -> 상세 정보 수집 완료 (on_tving_kr=True)

=== '춘화연애담' 검색 중 ===
  -> 선택된 TV id=224733, name